# Session 1 — From cultural text to an auditable LLM API call

**8th Baltic Summer School of Digital Humanities · BSSDH 2026**  
**Theme:** *Cultural Data Analytics and Meaning*  
**Length:** 90 minutes · Session 1 of 3

This session is deliberately narrow and empirical. By the end, you will have:

1. one practical mental model of LLM generation;
2. one successful OpenRouter model call;
3. one inspected JSON response;
4. one saved reproducibility record; and
5. one source-grounded extraction that you have checked against its source.

> **Assumed preparation:** you have reviewed [`workshop_session_0.ipynb`](workshop_session_0.ipynb), can run Python notebook cells, and have received an individual OpenRouter API key.


## Route through the session

| Time | Activity | Concrete result |
|---:|---|---|
| 0–8 min | Orientation and prerequisites | Notebook copy + API key ready |
| 8–20 min | One LLM mental model + 2026 landscape | Vocabulary for interpreting a call |
| 20–32 min | Three humanities examples | Research opportunities separated from validity claims |
| 32–43 min | API, OpenRouter, privacy | Request/response and data-flow model |
| 43–58 min | Setup + first model call | A source-grounded answer |
| 58–68 min | Inspect the response | Raw JSON + usage metadata |
| 68–77 min | Save provenance | A reproducibility record on disk |
| 77–87 min | Exercise | Extracted claims with evidence checks |
| 87–90 min | Debrief | Bridge to Session 2 |

**Before running code:** make your own Colab copy. A CPU runtime is sufficient. Run cells from top to bottom. Do not paste confidential, personal, or unpublished material into this workshop notebook.


<details>
<summary><strong>Instructors and assistants</strong> — expand for biographies</summary>

**Valdis Saulespurēns** works as a researcher and developer at the National Library of Latvia. He is also a lecturer at Riga Technical University, where he teaches Python, JavaScript, and other computer science subjects. Valdis specializes in machine learning and data analysis and enjoys transforming disordered data into structured knowledge. With more than 30 years of programming experience, he began his professional career writing programs for quantum scientists at the University of California, Santa Barbara. Before moving into teaching, he developed software for a radio broadcast equipment manufacturer. Valdis holds a Master's degree in Computer Science from the University of Latvia.

**Haralds Matulis** is a researcher and an organizer of this iteration of the Baltic Summer School of Digital Humanities. He has a background in digital humanities and is interested in the intersection of technology and humanities research. Haralds is dedicated to promoting digital literacy and innovation in the humanities.

**Zlata Batmanova** is a Master's student in Digital Humanities and a member of the DH student representative body at the University of Vienna. She previously completed a degree in Translation and Interpreting. Her research interests include NLP, LLMs, and their biases. She works as a student assistant at the Austrian Academy of Sciences (ACDH unit) on the project “Übersetzungsräume,” focusing on AI-assisted digitization of bibliographies and their quantitative processing.

**Melissa S. Lantzberg** is a Master's student in Digital Humanities and a member of the DH student representative body at the University of Vienna. She also studies English Language and Linguistics. Her latest DH project, conducted at the Natural History Museum of Vienna, focused on the orientation of early medieval churches. She is interested in learning and applying new DH methods across disciplines.

</details>


## 1 · One practical mental model

<img src="https://raw.githubusercontent.com/LNB-DH/BSSDH_2026_LLM_API_workshop/main/img/session_1/llm_token_prediction_mental_model.svg" alt="Source and task are tokenized, placed in a finite context, used to predict a next token, and repeated to produce text" width="100%">

A useful operational model is:

**source + instructions → tokens in a finite context → next-token probabilities → generated tokens**

Three terms are enough for today:

- A **token** is a model-specific unit of text. It may be a word, part of a word, punctuation, or whitespace. Token-to-word ratios vary by language, spelling, and OCR quality.
- A **context window** is the token budget available to a request. Instructions, source text, prior messages, and generated output all consume that budget. It is not permanent memory.
- **Generation** repeatedly selects a next token from a probability distribution. A fluent answer can therefore be unsupported, incomplete, or wrong.

The transformer architecture behind modern LLMs was introduced in [Vaswani et al., “Attention Is All You Need” (NeurIPS 2017)](https://papers.neurips.cc/paper/7181-attention-is-all-you-need). Published context limits are capacity ceilings, not evidence that a model will use every position equally well; see [Liu et al., “Lost in the Middle” (TACL 2024)](https://aclanthology.org/2024.tacl-1.9/).


### A dated 2026 model landscape

**Last verified: 4 August 2026.** Model catalogs, aliases, prices, and provider routes change. Treat this table as orientation and verify the live catalog before teaching or starting a study.

| Developer | Current examples | Published specification relevant to text research | Access note |
|---|---|---|---|
| OpenAI | GPT-5.6 Sol, Terra, Luna | 1.05M-token context; up to 128K output | Hosted API; three capability/cost tiers. [Official models](https://developers.openai.com/api/docs/models) |
| Anthropic | Claude Fable 5, Opus 5, Sonnet 5; Haiku 4.5 | Fable/Opus/Sonnet: 1M context and 128K output; Haiku: 200K/64K | Hosted API; model IDs are documented as pinned versions. [Official overview](https://platform.claude.com/docs/en/about-claude/models/overview) |
| Google | Gemini 3.6 Flash; Gemini 3.5 Flash-Lite | Flash-Lite: 1,048,576 input and 65,536 output tokens; structured outputs | Hosted API; Flash-Lite is the low-latency, high-throughput option used today. [Official model page](https://ai.google.dev/gemini-api/docs/models/gemini-3.5-flash-lite) |
| Meta | Llama 4 Scout; Llama 4 Maverick | Published context: 10M (Scout), 1M (Maverick) | Downloadable **open weights** under Meta's custom community license. [Official model card](https://github.com/meta-llama/llama-models/blob/main/models/llama4/MODEL_CARD.md) |
| Mistral AI | Mistral Large 3; Ministral 3 (3B/8B/14B) | Multimodal, versioned `v25.12` releases | Open-weight releases; verify each license and endpoint. [Official catalog](https://docs.mistral.ai/models/) |

**Do not equate “open weights” with “open source.”** The [Open Source AI Definition 1.0](https://opensource.org/ai/open-source-ai-definition) also considers access to the preferred form for modification, including data information and code. Likewise, a million-token context does not guarantee good retrieval or interpretation across a million tokens.

Today's requested OpenRouter model is defined once in a configuration cell: `google/gemini-3.5-flash-lite`. The [OpenRouter model page](https://openrouter.ai/google/gemini-3.5-flash-lite) and the returned response metadata—not this prose—are the operational record.


## 2 · Why use LLMs at the National Library of Latvia?

These examples motivate research questions; they are not claims that an LLM is automatically accurate.

| Case | Observation | Validity question | Minimum responsible check |
|---|---|---|---|
| Historical OCR and normalization | In internal exploratory work, models were more useful for modernization into contemporary spelling/script than for faithful diplomatic reconstruction. | Did the model correct OCR, or silently rewrite the source? | Retain the original; annotate a sample; report character/word errors and semantic changes. |
| Building a Latvian music-text corpus | Context-aware classification can reduce ambiguity that keyword filtering cannot resolve. | Which relevant texts were excluded, and which irrelevant texts were included? | Write annotation rules; label a sample; report a confusion matrix or precision/recall. |
| Transportation in Latvian novels | A study of about 460 novels compared Word2Vec and Gemini 1.5 for identifying transport terms. | Does a larger candidate list contain more valid evidence or simply more noise? | Human validation, error categories, and method comparison. |

The transportation study is now a 2026 Journal of Computational Literary Studies article: Eglāja-Kristsone, Baklāne, and Saulespurēns, [“Urban Transportation in Latvian Novels…”](https://doi.org/10.48694/jcls.4222), *JCLS* 5(1).

<img src="https://raw.githubusercontent.com/LNB-DH/BSSDH_2026_LLM_API_workshop/main/img/fraktur-normalization.JPG" alt="Internal National Library of Latvia example of OCR normalization" width="620">

<small>The OCR image is an internal workshop example, not a benchmark. The reusable lesson is to define the transformation and validation target before choosing a model.</small>


## 3 · From notebook to model—and back to evidence

<img src="https://raw.githubusercontent.com/LNB-DH/BSSDH_2026_LLM_API_workshop/main/img/session_1/source_to_claim_validation_loop.svg" alt="Notebook source and task travel through an API gateway to a model; the JSON response is validated and saved with provenance" width="100%">

An API call is a structured network exchange. The Python SDK constructs an HTTPS request with a JSON body; OpenRouter authenticates the key, routes the request to an available provider endpoint, and normalizes the provider's response into an OpenAI-compatible JSON shape.

| Chat interface | API workflow |
|---|---|
| Designed for interactive turns | Designed for structured requests and automation |
| Copy/paste is the usual unit of work | Loops can process a documented sample or corpus |
| Logs depend on the product UI | Your code can save prompts, parameters, outputs, and metadata |
| Convenient for exploration | Better suited to auditable pipelines |

The API improves **auditability**, not automatic reproducibility or validity. Providers can update infrastructure, gateways can route differently, and generation can vary. Record what happened and evaluate outputs against sources or labeled data.

Reference: [OpenRouter quickstart and OpenAI SDK configuration](https://openrouter.ai/docs/quickstart).


### Privacy and security checkpoint

- Treat an API key like a password. This notebook requests it with `getpass`; it never writes the key to the notebook, `.env`, or the reproducibility record.
- OpenRouter states that prompt/response storage is opt-in on its own service, while request metadata such as token counts and latency is stored. Individual model providers have separate retention and training policies.
- Zero Data Retention (ZDR) routing can restrict inference to qualifying endpoints, but it does not cover every optional tool or plugin.
- Do not send personal, confidential, copyrighted-at-scale, culturally sensitive, or unpublished research data unless you have authority and have reviewed the relevant institutional and provider policies.

Authoritative policy pages: [OpenRouter data collection](https://openrouter.ai/docs/guides/privacy/data-collection), [provider logging](https://openrouter.ai/docs/guides/privacy/provider-logging), and [ZDR](https://openrouter.ai/docs/guides/features/zdr).


## 4 · Set up the runtime

Run the next cell once. It installs a compatible OpenAI Python SDK, which OpenRouter supports as an OpenAI-compatible client. The upper bound avoids an untested future major release during the workshop; the exact installed version will be recorded later.


In [ ]:
%pip install -q "openai>=1.58,<3"


In [ ]:
import getpass
import hashlib
import json
import platform
from datetime import datetime, timezone
from pathlib import Path

import openai
from openai import OpenAI

OPENROUTER_API_KEY = getpass.getpass("Paste your OpenRouter API key (hidden): ").strip()
if not OPENROUTER_API_KEY:
    raise ValueError("No API key was entered.")

OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"
MODEL_ID = "google/gemini-3.5-flash-lite"
MODEL_LAST_VERIFIED = "2026-08-04"

client = OpenAI(
    base_url=OPENROUTER_BASE_URL,
    api_key=OPENROUTER_API_KEY,
    default_headers={
        "HTTP-Referer": "https://digitalhumanities.lv/en/bssdh/2026/",
        "X-OpenRouter-Title": "BSSDH 2026 LLM API Workshop",
        "X-OpenRouter-Metadata": "enabled",
    },
)

print(f"OpenAI SDK: {openai.__version__}")
print(f"Requested model: {MODEL_ID} (verified {MODEL_LAST_VERIFIED})")
print("Client configured; no model request has been sent yet.")


## 5 · Make one successful, source-grounded call

First, make the request visible. The system message sets the evidence rule, the user message contains both the source and task, and `max_completion_tokens` limits the generated portion. We intentionally omit deprecated sampling parameters such as `temperature` for this model family.


In [ ]:
WORKSHOP_SOURCE = """
In this workshop, participants will learn how to access large language models via API
and utilize them for bulk data analysis using Python. Through practical examples, we
will explore prompt engineering for concept mining and named entity recognition in
textual data. We will also examine challenges in historical digitized texts, including
OCR errors, and consider error correction and translation. The workshop is designed
for researchers, data analysts, and professionals in text analysis, digital humanities,
and computational linguistics. Only basic familiarity with Python is required.
""".strip()

SYSTEM_MESSAGE = (
    "You are a careful research assistant. Use only the supplied source. "
    "Do not add outside facts. If the source does not support a requested claim, "
    "write NOT IN SOURCE."
)

FIRST_TASK = """
In no more than 120 words, report:
1. the computational method participants will learn;
2. two named text-analysis tasks;
3. one challenge involving historical material; and
4. the stated prerequisite.
Include a short exact evidence phrase after each item.
""".strip()

messages = [
    {"role": "system", "content": SYSTEM_MESSAGE},
    {
        "role": "user",
        "content": f"SOURCE:\n{WORKSHOP_SOURCE}\n\nTASK:\n{FIRST_TASK}",
    },
]

request_preview = {
    "model": MODEL_ID,
    "messages": messages,
    "max_completion_tokens": 350,
}

# This is the JSON-shaped request body. It contains no API key.
print(json.dumps(request_preview, indent=2, ensure_ascii=False))


In [ ]:
CALL_STARTED_AT_UTC = datetime.now(timezone.utc).isoformat()

try:
    response = client.chat.completions.create(**request_preview)
except Exception:
    print("The call failed. Check the key, credits, model page, and OpenRouter status/error guide.")
    print("https://openrouter.ai/docs/api/reference/errors-and-debugging")
    raise

CALL_FINISHED_AT_UTC = datetime.now(timezone.utc).isoformat()
FIRST_TEXT = response.choices[0].message.content or ""

if not FIRST_TEXT.strip():
    raise RuntimeError("The API returned no generated text.")

print("✓ Successful model call\n")
print(FIRST_TEXT)


## 6 · Inspect the JSON response

The displayed answer is only one nested field. Inspect the complete SDK object as JSON before writing extraction code. Common fields include:

- `id`: request/response identifier;
- `model`: the model identifier returned by the service;
- `choices[0].message.content`: generated text;
- `choices[0].finish_reason`: why generation stopped;
- `usage`: input, output, and total token counts, with possible provider-specific additions;
- `openrouter_metadata`: routing information when the requested endpoint returns it.


In [ ]:
response_json = response.model_dump(mode="json")
print(json.dumps(response_json, indent=2, ensure_ascii=False, default=str))


In [ ]:
choice = response_json["choices"][0]
usage = response_json.get("usage") or {}

print("Requested model :", MODEL_ID)
print("Returned model  :", response_json.get("model"))
print("Response ID     :", response_json.get("id"))
print("Finish reason   :", choice.get("finish_reason"))
print("Input tokens    :", usage.get("prompt_tokens"))
print("Output tokens   :", usage.get("completion_tokens"))
print("Total tokens    :", usage.get("total_tokens"))
print("Reported cost   :", usage.get("cost", "not returned"))
print("Router metadata :", response_json.get("openrouter_metadata", "not returned"))


## 7 · Save one reproducibility record

A record should be sufficient to understand and rerun the call: when it happened, the endpoint, requested and returned model, exact messages, non-secret parameters, source hash, output, usage, and software versions.

This improves auditability; it does **not** promise byte-identical output. Model routing, infrastructure, and nondeterministic generation can still differ. Never store the API key.


In [ ]:
source_sha256 = hashlib.sha256(WORKSHOP_SOURCE.encode("utf-8")).hexdigest()

reproducibility_record = {
    "record_schema": "bssdh.llm-call.v1",
    "notebook_revision": "BSSDH 2026 Session 1",
    "call_started_at_utc": CALL_STARTED_AT_UTC,
    "call_finished_at_utc": CALL_FINISHED_AT_UTC,
    "source": {
        "label": "BSSDH 2026 workshop description (workshop-provided excerpt)",
        "sha256": source_sha256,
        "characters": len(WORKSHOP_SOURCE),
    },
    "request": {
        "endpoint": f"{OPENROUTER_BASE_URL}/chat/completions",
        "requested_model": MODEL_ID,
        "model_last_verified": MODEL_LAST_VERIFIED,
        "messages": messages,
        "parameters": {"max_completion_tokens": 350},
    },
    "response": {
        "id": response_json.get("id"),
        "returned_model": response_json.get("model"),
        "system_fingerprint": response_json.get("system_fingerprint"),
        "finish_reason": choice.get("finish_reason"),
        "text": FIRST_TEXT,
        "usage": usage,
        "openrouter_metadata": response_json.get("openrouter_metadata"),
    },
    "environment": {
        "python": platform.python_version(),
        "openai_sdk": openai.__version__,
        "platform": platform.platform(),
    },
    "validation": {
        "status": "not_yet_validated",
        "note": "Generated claims still require comparison with the source.",
    },
}

RECORD_PATH = Path("session_1_reproducibility_record.json")
RECORD_PATH.write_text(
    json.dumps(reproducibility_record, indent=2, ensure_ascii=False, default=str),
    encoding="utf-8",
)

print(f"Saved: {RECORD_PATH.resolve()}")
print(json.dumps(reproducibility_record, indent=2, ensure_ascii=False, default=str))


## 8 · Source-grounded exercise: extraction plus an abstention test

The source below is an instructor-written summary of the 2026 transport study, not a quotation from the article. The task deliberately asks for one value that the summary does not contain. A strong result must abstain instead of guessing.

**Work in pairs (10 minutes):**

1. Read the source and predict which field must be `not_in_source`.
2. Run the call.
3. Parse the model's JSON text.
4. Check whether each evidence string occurs *exactly* in the source.
5. Classify any failure as extraction, evidence, formatting, or unsupported inference.

Full study: Eglāja-Kristsone, Baklāne, and Saulespurēns (2026), [DOI 10.48694/jcls.4222](https://doi.org/10.48694/jcls.4222).


In [ ]:
def ask_about_source(source_text, task, *, model=MODEL_ID, max_completion_tokens=500):
    exercise_messages = [
        {"role": "system", "content": SYSTEM_MESSAGE},
        {
            "role": "user",
            "content": f"SOURCE:\n{source_text}\n\nTASK:\n{task}",
        },
    ]
    return client.chat.completions.create(
        model=model,
        messages=exercise_messages,
        max_completion_tokens=max_completion_tokens,
    )


TRANSPORT_STUDY_SOURCE = """
A 2026 study by Eva Eglāja-Kristsone, Anda Baklāne, and Valdis Saulespurēns
examines how urban transportation in Latvian novels reflects modernization and social
change. The research uses the Corpus of Early Latvian Novels, containing approximately
460 works from 1879 to 1940. It compares Word2Vec with Gemini 1.5 to identify candidate
transportation terms and then analyzes how vehicle references are distributed across
the corpus.
""".strip()

EXERCISE_TASK = """
Return ONLY a valid JSON array. Create one object for each field:
research_focus, corpus_size, corpus_period, methods, accuracy_percentage, best_model.
Each object must have exactly these keys:
- field
- value (string or null)
- evidence (an exact substring of SOURCE, or null)
- status (supported or not_in_source)
Do not infer missing values.
""".strip()

exercise_response = ask_about_source(TRANSPORT_STUDY_SOURCE, EXERCISE_TASK)
EXERCISE_TEXT = exercise_response.choices[0].message.content or ""
print(EXERCISE_TEXT)


In [ ]:
def parse_json_text(text):
    """Remove an optional Markdown fence, then parse JSON."""
    cleaned = text.strip()
    if cleaned.startswith("```"):
        lines = cleaned.splitlines()
        cleaned = "\n".join(lines[1:-1]).strip()
    return json.loads(cleaned)


extracted_items = parse_json_text(EXERCISE_TEXT)

for item in extracted_items:
    evidence = item.get("evidence")
    status = item.get("status")
    if status == "supported":
        evidence_is_exact = (
            isinstance(evidence, str)
            and bool(evidence)
            and evidence in TRANSPORT_STUDY_SOURCE
        )
        value_matches_status = item.get("value") is not None
    elif status == "not_in_source":
        evidence_is_exact = evidence is None
        value_matches_status = item.get("value") is None
    else:
        evidence_is_exact = False
        value_matches_status = False
    print(
        f"{item.get('field', '?'):>20} | "
        f"exact evidence: {evidence_is_exact!s:<5} | "
        f"status/value consistent: {value_matches_status}"
    )

print("\nExpected abstentions: accuracy_percentage and best_model")


### Debrief: what counts as success?

- The model call completed, but completion alone did not establish validity.
- The API envelope was JSON; model-generated content was a string that we separately parsed as JSON.
- An exact evidence substring is a useful automated check, but interpretation still requires a researcher.
- “Not in source” is a valid and often desirable research output.
- Agreement between two LLMs is not ground truth. Validate against the source, an annotated sample, or a defensible external reference.

**Bridge to Session 2:** we will reuse this request → response → validation pattern for repeated text-analysis tasks and corpus-scale workflows in [`workshop_session_2.ipynb`](workshop_session_2.ipynb).


## Further information and authoritative references

**Models and APIs (live documentation)**

- [OpenRouter quickstart](https://openrouter.ai/docs/quickstart), [models](https://openrouter.ai/models), [errors and debugging](https://openrouter.ai/docs/api/reference/errors-and-debugging), and [router metadata](https://openrouter.ai/docs/guides/features/router-metadata)
- [OpenAI model catalog](https://developers.openai.com/api/docs/models)
- [Anthropic model overview](https://platform.claude.com/docs/en/about-claude/models/overview)
- [Google Gemini model catalog](https://ai.google.dev/gemini-api/docs/models)
- [Meta Llama 4 model card](https://github.com/meta-llama/llama-models/blob/main/models/llama4/MODEL_CARD.md)
- [Mistral model catalog](https://docs.mistral.ai/models/)

**Method, documentation, and responsible interpretation**

- Vaswani et al. (2017), [“Attention Is All You Need”](https://papers.neurips.cc/paper/7181-attention-is-all-you-need)
- Liu et al. (2024), [“Lost in the Middle: How Language Models Use Long Contexts”](https://aclanthology.org/2024.tacl-1.9/)
- Mitchell et al. (2019), [“Model Cards for Model Reporting”](https://doi.org/10.1145/3287560.3287596)
- Wilkinson et al. (2016), [“The FAIR Guiding Principles for scientific data management and stewardship”](https://doi.org/10.1038/sdata.2016.18)
- [Open Source AI Definition 1.0](https://opensource.org/ai/open-source-ai-definition)
- [Google Colab FAQ](https://research.google.com/colaboratory/faq.html)

**Workshop research example**

- Eglāja-Kristsone, Baklāne, and Saulespurēns (2026), [“Urban Transportation in Latvian Novels or Why Do You Use a 19th-Century Horse-drawn Cab When You Have a 20th-Century Taxi?”](https://doi.org/10.48694/jcls.4222), *Journal of Computational Literary Studies*, 5(1).

> **Instructor preflight on workshop day:** verify the OpenRouter model slug, access, provider policy, and current price; run the notebook top to bottom with a fresh participant-style key; then clear all outputs before distribution.
